# Firth logistic regression: seizure types, LGS and outcome

Runs on one **unit** (`UNIT`, the first code cell; see `cc_stats/units.py`): `types`, the seizure-type groups, or `codes`, the individual ILAE 2017 codes with at least 10 patients before surgery. `types/Firth_results.ipynb` is the source; `codes/Firth_results.ipynb` is generated from it by `analysis/unit_notebooks.py` (only `UNIT` and the notes at the end differ), so edit the types notebook and regenerate. Below, "type" means a group of the unit.

Outcome (see `cc_stats/outcomes.py`): Engel, `engel_improved` (1 = Engel I–III, 0 = Engel IV), and ILAE, `ilae_improved` (1 = ILAE 1–4, 0 = ILAE 5–6). Every table and figure is shown for both outcomes, Engel then ILAE; the R scripts fit both. Cohort: patients with the outcome, LGS and pre-op seizure types recorded. The cohort size and number not improved are printed below.

Models are fitted in R with `logistf`: `firths_analysis.R` writes `firths_results.csv` and `firths_results_lgs_adjusted.csv`. This notebook re-runs it and presents the results. Sex and age at surgery are in `0_cohort/patient_characteristics.ipynb`.

With so few non-improved patients, every model has one predictor, or two in the LGS-adjusted sensitivity check.

1. **Univariable:** `improved ~ type` for each type, plus `improved ~ lgs`
2. **LGS-adjusted:** `improved ~ type + lgs` for each type, reporting both the type and the LGS estimates

Odds ratios have profile-likelihood 95% CIs. Holm correction is applied within each set of tests, and within each outcome.

In [ ]:
# The unit this notebook runs on: 'types' or 'codes' (cc_stats/units.py).
# Set here only; the codes notebook is generated with 'codes' (analysis/unit_notebooks.py).
UNIT = 'codes'

In [ ]:
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display
from scipy.stats.contingency import odds_ratio
from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import ANALYSIS_CSV, PROJECT_ROOT, results_dir
from cc_stats.plots import (right_of, forest, INK, PAIR_A, PAIR_B, paired_rate_plot, save_figure,
                            setup_or_axis)
from cc_stats.rscript import find_rscript
from cc_stats.units import unit as get_unit


RSCRIPT = find_rscript()
print(f"Using {RSCRIPT}")
run = subprocess.run([RSCRIPT, "analysis/1_improvement/firths_analysis.R", UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f"firths_analysis.R failed:\n{run.stderr}")

# The R scripts fit both outcomes; split each result by outcome.
FOLDER = f"1_improvement/{UNIT}"
RESULTS = results_dir(FOLDER)
def by_outcome(f):
    table = pd.read_csv(RESULTS / f)
    return {outcome: table.query("outcome == @outcome").reset_index(drop=True) for outcome in OUTCOMES}
uni, adj = (by_outcome(f) for f in ("firths_results.csv", "firths_results_lgs_adjusted.csv"))

analysis = pd.read_csv(ANALYSIS_CSV, index_col="subject_id")
unit = get_unit(UNIT, analysis)
# 'type' or 'code', for figure titles and labels
WORD = 'type' if UNIT == 'types' else 'code'
has_cols = [unit.pre(t) for t in unit.groups]
# Figures grow with the number of rows (drawn for the 10 type groups)
tall = max(1, len(unit.groups) / 10)
patients = {}
for outcome, O in OUTCOMES.items():
    patients[outcome] = analysis.dropna(subset=[O["improved"], "lgs"] + has_cols)
    p = patients[outcome]
    print(f"{O['name']}: n = {len(p)}, {int((p[O['improved']] == 0).sum())} not improved")

In [ ]:
def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'

def fmt_num(v):
    return f'{v:.3f}' if v < 0.01 else f'{v:.2f}'

def fmt_or(o, lo, hi):
    return f'{fmt_num(o)} ({fmt_num(lo)}–{fmt_num(hi)})'


## 1. Univariable models

One model per predictor: `improved ~ predictor`. *Improved* is the observed count in each group. The figure shows the same observed rates with Wilson 95% CIs, and improved / total for each group on the right.

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    u = uni[outcome].sort_values('p').reset_index(drop=True)
    display(pd.DataFrame({
        'Predictor': u.predictor,
        'With / without': u.n_with.astype(str) + ' / ' + u.n_without.astype(str),
        'Improved, without': u.improved_without.astype(str) + '/' + u.n_without.astype(str),
        'Improved, with': u.improved_with.astype(str) + '/' + u.n_with.astype(str),
        'OR (95% CI)': [fmt_or(*r) for r in u[['OR', 'lo', 'hi']].to_numpy()],
        'p': u.p.map(fmt_p),
        'Holm p': u.p_holm.map(fmt_p),
    }).style.hide(axis='index'))

In [ ]:
# Observed improvement with vs without each predictor, Wilson 95% CIs. These are
# the observed counts from the table above, not the model's probabilities.
for outcome, O in OUTCOMES.items():
    heading(outcome)
    d = uni[outcome].sort_values('OR').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9.5, 7 * tall))
    paired_rate_plot(ax, d.predictor,
                     top=('with predictor', d.improved_with, d.n_with, PAIR_A),
                     bottom=('without predictor', d.improved_without, d.n_without, PAIR_B))
    ax.set_xlabel(f"Observed improvement, {O['improved_label']} (Wilson 95% CI)")
    ax.set_title(f"{O['name']}: observed improvement with vs without each predictor", loc='left', pad=28)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
    plt.tight_layout()
    save_figure(FOLDER, f'observed_improvement_with_without_{outcome}')
    plt.show()

## 2. LGS-adjusted models: `improved ~ type + lgs`

### 2a. Seizure-type odds ratios, adjusted for LGS

The unadjusted OR from section 1 is shown next to the adjusted OR for comparison. The unadjusted Holm p is the one from section 1, corrected across its models (every type plus LGS); the adjusted Holm p is corrected across the `type + lgs` models.

Firth's correction gives a finite OR even when everyone with a type improved (pure_focal on both outcomes, for types), where the observed odds ratio would be infinite.

In [ ]:
# Each type's LGS-adjusted model next to its unadjusted one from section 1
a = {outcome: adj[outcome].merge(uni[outcome][['predictor', 'OR', 'lo', 'hi', 'p', 'p_holm', 'n_with', 'improved_with', 'n_without', 'improved_without']],
                                 left_on='seizure_type', right_on='predictor')
                          .sort_values('type_p').reset_index(drop=True)
     for outcome in OUTCOMES}

for outcome in OUTCOMES:
    heading(outcome)
    t = a[outcome]
    display(pd.DataFrame({
        'Seizure type': t.seizure_type,
        'Unadjusted OR (95% CI)': [fmt_or(*r) for r in t[['OR', 'lo', 'hi']].to_numpy()],
        'Unadjusted p': t.p.map(fmt_p),
        'Unadjusted Holm p': t.p_holm.map(fmt_p),
        'LGS-adjusted OR (95% CI)': [fmt_or(*r) for r in t[['type_OR', 'type_lo', 'type_hi']].to_numpy()],
        'Adjusted p': t.type_p.map(fmt_p),
        'Adjusted Holm p': t.type_p_holm.map(fmt_p),
    }).style.hide(axis='index'))

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    u = uni[outcome]
    yorder = u[u.predictor != 'lgs'].sort_values('OR').predictor.tolist()
    d = a[outcome].set_index('seizure_type').loc[yorder].reset_index()

    fig, ax = plt.subplots(figsize=(10.5, 6.5 * tall))
    forest(ax, d, ['OR', 'lo', 'hi'], offset=0.18, color=PAIR_A, label='unadjusted')
    forest(ax, d, ['type_OR', 'type_lo', 'type_hi'], offset=-0.18, color=PAIR_B, label='adjusted for LGS')
    setup_or_axis(ax, d.seizure_type,
                  f"{OUTCOMES[outcome]['name']}: seizure-{WORD} odds ratio, unadjusted vs adjusted for LGS")
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False)
    # Beside each row, a fixed distance right of the plot: patients with / without
    # the type (the same for both models), then each model's p and Holm p on its
    # own line, in its colour (unadjusted teal above, adjusted orange below)
    at = lambda x, **kw: dict(transform=right_of(ax, x), va='center', ha='center', fontsize=9, **kw)
    for x, header in ((0.5, 'n with /\nwithout'), (1.35, 'p'), (2.05, 'Holm p')):
        ax.text(1, len(d) - 0.3, header, **at(x, fontweight='bold'))
    for k, r in d.iterrows():
        ax.text(1, k, f'{r.n_with:.0f} / {r.n_without:.0f}', **at(0.5))
        for offset, color, p, p_holm in ((0.18, PAIR_A, r.p, r.p_holm), (-0.18, PAIR_B, r.type_p, r.type_p_holm)):
            ax.text(1, k + offset, fmt_p(p), **at(1.35, color=color))
            ax.text(1, k + offset, fmt_p(p_holm), **at(2.05, color=color))
    plt.tight_layout()
    save_figure(FOLDER, f'firth_types_unadjusted_vs_lgs_adjusted_{outcome}')
    plt.show()

In [ ]:
# The LGS-adjusted ORs on their own, for slides: Engel (teal, above) and ILAE
# (orange, below) on one plot, with each outcome's group sizes and Holm p on its
# own line, in its colour.
yorder = uni['engel'].query("predictor != 'lgs'").sort_values('OR').predictor.tolist()  # same rows for both
fig, ax = plt.subplots(figsize=(10, 7 * tall))
for outcome, offset in (('engel', 0.17), ('ilae', -0.17)):
    d = a[outcome].set_index('seizure_type').loc[yorder].reset_index()
    forest(ax, d, ['type_OR', 'type_lo', 'type_hi'], offset=offset,
           color={'engel': PAIR_A, 'ilae': PAIR_B}[outcome], label=OUTCOMES[outcome]['name'])
setup_or_axis(ax, yorder, f'Seizure-{WORD} odds ratio, adjusted for LGS')
ax.set_ylim(-0.6, len(yorder) - 0.4)
columns = [(0.6, 'n with /\nwithout'), (1.5, 'Holm p')]
at = lambda x, **kw: dict(transform=right_of(ax, x), va='center', ha='center', fontsize=9, **kw)
for x, header in columns:
    ax.text(1, len(yorder) - 0.3, header, **at(x, fontweight='bold'))
for outcome, offset in (('engel', 0.17), ('ilae', -0.17)):
    d = a[outcome].set_index('seizure_type').loc[yorder]
    color = {'engel': PAIR_A, 'ilae': PAIR_B}[outcome]
    for k, r in enumerate(d.itertuples()):
        for (x, _), value in zip(columns, (f'{r.n_with:.0f} / {r.n_without:.0f}', fmt_p(r.type_p_holm))):
            ax.text(1, k + offset, value, **at(x, color=color))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.09), ncol=2, frameon=False)
plt.tight_layout()
save_figure(FOLDER, 'firth_types_lgs_adjusted')
plt.show()

### 2b. LGS odds ratio within each seizure-type model

**Does LGS look the way it does only because of the seizure types LGS patients tend to have?** No: adjusting for any one seizure type barely moves the LGS odds ratio (the range is printed below the table), so seizure type doesn't explain the LGS result.

Each row is the LGS coefficient from a different `type + lgs` model, i.e. the LGS effect with that type held constant. The first row is LGS on its own (section 1), which is the reference. The adjusted estimates come from the same patients, so they should be compared with the reference, not averaged.

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    lgs_alone = uni[outcome].set_index('predictor').loc['lgs']
    b = a[outcome].sort_values('lgs_OR')
    display(pd.concat([
        pd.DataFrame({'Model': ['lgs alone'],
                      'LGS OR (95% CI)': [fmt_or(lgs_alone.OR, lgs_alone.lo, lgs_alone.hi)],
                      'p': [fmt_p(lgs_alone.p)],
                      'Change vs alone': ['']}),
        pd.DataFrame({'Model': '+ ' + b.seizure_type,
                      'LGS OR (95% CI)': [fmt_or(*r) for r in b[['lgs_OR', 'lgs_lo', 'lgs_hi']].to_numpy()],
                      'p': b.lgs_p.map(fmt_p),
                      'Change vs alone': (b.lgs_OR / lgs_alone.OR - 1).map(lambda v: f'{v:+.0%}')}),
    ]).style.hide(axis='index'))
    print(f'LGS alone: OR {lgs_alone.OR:.2f}; adjusted for one type at a time: {b.lgs_OR.min():.2f} to {b.lgs_OR.max():.2f}')

In [ ]:
# LGS's OR on its own (top row) and within each type's model, Engel (teal,
# above) and ILAE (orange, below) on one plot, with a dotted line at each
# outcome's LGS-alone OR. The message is how little the
# OR moves down the rows, so the exact values stay in the table above.
types = a['engel'].sort_values('lgs_OR', ascending=False).seizure_type.tolist()  # same rows for both


def lgs_rows(outcome):
    lgs_alone = uni[outcome].set_index('predictor').loc['lgs']
    b = a[outcome].set_index('seizure_type').loc[types]
    return pd.concat([b[['lgs_OR', 'lgs_lo', 'lgs_hi']].set_axis(['OR', 'lo', 'hi'], axis=1),
                      pd.DataFrame({'OR': [lgs_alone.OR], 'lo': [lgs_alone.lo], 'hi': [lgs_alone.hi]})]
                     ).reset_index(drop=True)


fig, ax = plt.subplots(figsize=(9.5, 7 * tall))
for outcome, offset in (('engel', 0.17), ('ilae', -0.17)):
    rows = lgs_rows(outcome)
    color = {'engel': PAIR_A, 'ilae': PAIR_B}[outcome]
    forest(ax, rows, ['OR', 'lo', 'hi'], offset=offset, color=color, label=OUTCOMES[outcome]['name'])
    ax.axvline(rows['OR'].iloc[-1], color=color, lw=1.2, ls=':')
labels = ['+ ' + t for t in types] + ['LGS alone']
setup_or_axis(ax, labels, f'LGS odds ratio, alone and adjusted for each seizure {WORD}',
              ticks=(0.1, 0.25, 0.5, 1, 2, 4, 10))
# Beside each row, on each outcome's line and in its colour: patients with / without
# LGS (top row) or with / without the seizure type that model adjusts for
ax.text(1, len(labels) - 0.35, 'n with / without', transform=right_of(ax, 0.75), va='center', ha='center',
        fontsize=9, fontweight='bold')
for outcome, offset in (('engel', 0.17), ('ilae', -0.17)):
    n = uni[outcome].set_index('predictor').loc[types + ['lgs'], ['n_with', 'n_without']]
    for y, (n_with, n_without) in enumerate(n.to_numpy()):
        ax.text(1, y + offset, f'{n_with:.0f} / {n_without:.0f}', transform=right_of(ax, 0.75), va='center',
                ha='center', fontsize=9, color={'engel': PAIR_A, 'ilae': PAIR_B}[outcome])
ax.get_yticklabels()[-1].set_fontweight('bold')
ax.axhline(len(labels) - 1.5, color='#bbbbbb', lw=1)  # LGS alone above, adjusted models below
ax.legend(handles=ax.get_legend_handles_labels()[0]
          + [Line2D([], [], color=INK, lw=1.2, ls=':', label='LGS alone (each outcome)')],
          loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
plt.tight_layout()
save_figure(FOLDER, 'firth_lgs_adjusted_for_each_type')
plt.show()

### 2c. Observed improvement by type and LGS

For each seizure type, patients split four ways by whether they have that type and whether they have LGS: improved / patients in each group, as counted, not modelled. Groups with only a few patients give unreliable percentages. The figure shows the same counts in two panels, patients without LGS and patients with LGS, so you can see whether a type goes with improvement in both.

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    p = patients[outcome]
    has_type = p[has_cols].set_axis(unit.groups, axis=1)
    improved = p[OUTCOMES[outcome]['improved']]

    def cell(has, lgs):
        group = improved[has & (p.lgs == lgs)]
        return f'{int(group.sum())}/{len(group)} ({group.mean():.0%})' if len(group) else '0/0'

    rows = []
    for t in a[outcome].seizure_type:
        has = has_type[t] == 1
        rows.append({
            'Seizure type': t,
            'Neither': cell(~has, 0),
            'Type only': cell(has, 0),
            'LGS only': cell(~has, 1),
            'Type + LGS': cell(has, 1),
        })
    display(pd.DataFrame(rows).style.hide(axis='index'))

In [ ]:
# The same counts as the table, as two panels: within patients without LGS and
# within patients with LGS, improvement with vs without each type.
for outcome, O in OUTCOMES.items():
    heading(outcome)
    p = patients[outcome]
    improved = p[O['improved']]
    types = uni[outcome].query("predictor != 'lgs'").sort_values('OR').predictor.tolist()

    fig, axes = plt.subplots(1, 2, figsize=(15, 7 * tall), sharey=True)
    for ax, lgs, name in ((axes[0], 0, 'Without LGS'), (axes[1], 1, 'With LGS')):
        group = p.lgs == lgs
        counts = {g: ([int(improved[group & (p[unit.pre(t)] == h)].sum()) for t in types],
                      [int((group & (p[unit.pre(t)] == h)).sum()) for t in types])
                  for g, h in (('with', 1), ('without', 0))}
        paired_rate_plot(ax, types, top=(f'with {WORD}', *counts['with'], PAIR_A),
                         bottom=(f'without {WORD}', *counts['without'], PAIR_B))
        ax.set_title(f'{name} (n={int(group.sum())})', loc='left', pad=10)
        ax.set_xlabel(f"Observed improvement, {O['improved_label']} (Wilson 95% CI)")
    axes[0].legend(loc='lower left', bbox_to_anchor=(0, 1.06), ncol=2, frameon=False, handletextpad=0.3)
    fig.suptitle(f"{O['name']}: improvement with vs without each seizure {WORD}, by LGS", x=0.01, ha='left')
    plt.tight_layout(w_pad=12)
    save_figure(FOLDER, f'observed_improvement_by_type_and_lgs_{outcome}')
    plt.show()

## Notes for interpretation

Checked against the v2 run on 2026-10-21. The cohort is the same as for types: Engel n = 95 (16 not improved), ILAE n = 88 (23 not improved), so the events per predictor are the same too (16 univariable, 8 LGS-adjusted on Engel). The 14 codes are those with at least 10 of the 100 patients with pre-op types; in the model cohort, focal motor clonic (19) has 9 on Engel and 8 on ILAE.

- **Generalized motor myoclonic (code 5) is the one result that survives Holm, on ILAE:** 23/24 improved with it vs 42/64 without, OR 8.3 (1.9–78), Holm p = 0.037; adjusted for LGS, OR 9.7, Holm p = 0.018. On Engel the direction is the same (25/26 vs 54/69, OR 4.8, 1.1–46, p = 0.035) but Holm p is 0.53 (0.45 adjusted). This is the myoclonic type's result, a little sharper: the type adds focal motor myoclonic (code 22, 5 patients), and its adjusted ILAE Holm p is 0.039.
- **Within LGS** (2c), everyone with generalized myoclonic improved on Engel (15/15, vs 22/30 without).
- **Focal non-motor impaired awareness (26) and behaviour arrest (29)** point towards improvement on ILAE (17/18 and 16/17 improved; p 0.02 and 0.03), not after Holm (0.32, 0.41). They are largely the same seizures described twice (awareness and feature, phi 0.97), so they are one signal, not two.
- **Separation:** everyone with atypical absence (11) improved on Engel (12/12), so its CI runs to about 800.
- **No code goes with a worse outcome** after correction. Generalized atonic (8) is the lowest on Engel (OR 0.42, 0.14–1.2, p = 0.11).
- **LGS:** adjusting for any one code barely moves the LGS OR: 0.76–1.02 on Engel (alone 0.88), 0.41–0.63 on ILAE (alone 0.53).
- **Overlap:** codes overlap more than the type groups do (a patient has a median of 4), so the 14 tests are far from independent; Holm is conservative here.